# 03. Очистка и предварительная обработка дневных данных ОФЗ

Ноутбук берёт CSV-файлы, скачанные с Московской биржи, объединяет их, приводит типы данных, удаляет критические ошибки, отмечает и заменяет выбросы.

Месячная агрегация в этом варианте проекта не используется: на выходе сохраняются только дневные данные и отчёты по качеству данных.

Основные выходные файлы сохраняются в `data/processed/`, отчёты по качеству данных — в `data/summary/`.


## 1. Импорт библиотек и параметры


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

START_DATE = pd.Timestamp("2016-05-03")
END_DATE = pd.Timestamp("2026-05-03")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SUMMARY_DIR = DATA_DIR / "summary"

INPUT_DIR = RAW_DIR / "ofz_moex_csv_2016_2026"
OUTPUT_DIR = PROCESSED_DIR
PER_BOND_CLEAN_DIR = OUTPUT_DIR / "by_bond_daily"
REPORTS_DIR = SUMMARY_DIR

for path in [OUTPUT_DIR, PER_BOND_CLEAN_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Input dir:", INPUT_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Reports dir:", REPORTS_DIR.resolve())


Input dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/ofz_moex_csv_2016_2026
Output dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed
Reports dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary


## 2. Метаданные по выбранным ОФЗ

Здесь фиксируются названия выпусков, тип купона и исходная группа отбора.


In [2]:
BONDS = [
    # Краткосрочные ОФЗ-ПД в текущей выборке
    {"SECID": "SU26219RMFS4", "ISSUE_NAME": "ОФЗ-ПД 26219", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "short_fixed_selected"},
    {"SECID": "SU26226RMFS9", "ISSUE_NAME": "ОФЗ-ПД 26226", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "short_fixed_selected"},
    {"SECID": "SU26207RMFS9", "ISSUE_NAME": "ОФЗ-ПД 26207", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "short_fixed_selected"},

    # Среднесрочные ОФЗ-ПД в текущей выборке
    {"SECID": "SU26232RMFS7", "ISSUE_NAME": "ОФЗ-ПД 26232", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "medium_fixed_selected"},
    {"SECID": "SU26236RMFS8", "ISSUE_NAME": "ОФЗ-ПД 26236", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "medium_fixed_selected"},
    {"SECID": "SU26242RMFS6", "ISSUE_NAME": "ОФЗ-ПД 26242", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "medium_fixed_selected"},

    # Долгосрочные ОФЗ-ПД в текущей выборке
    {"SECID": "SU26240RMFS0", "ISSUE_NAME": "ОФЗ-ПД 26240", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "long_fixed_selected"},
    {"SECID": "SU26243RMFS4", "ISSUE_NAME": "ОФЗ-ПД 26243", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "long_fixed_selected"},
    {"SECID": "SU26238RMFS4", "ISSUE_NAME": "ОФЗ-ПД 26238", "COUPON_TYPE": "fixed", "ANALYSIS_GROUP": "long_fixed_selected"},

    # ОФЗ-ПК с плавающим купоном
    {"SECID": "SU29024RMFS5", "ISSUE_NAME": "ОФЗ-ПК 29024", "COUPON_TYPE": "floating", "ANALYSIS_GROUP": "floating_coupon"},
    {"SECID": "SU29025RMFS2", "ISSUE_NAME": "ОФЗ-ПК 29025", "COUPON_TYPE": "floating", "ANALYSIS_GROUP": "floating_coupon"},
    {"SECID": "SU29026RMFS0", "ISSUE_NAME": "ОФЗ-ПК 29026", "COUPON_TYPE": "floating", "ANALYSIS_GROUP": "floating_coupon"},
]

bond_meta = pd.DataFrame(BONDS)
BOND_META = bond_meta.set_index("SECID").to_dict("index")

bond_meta


,SECID,ISSUE_NAME,COUPON_TYPE,ANALYSIS_GROUP
0,SU26219RMFS4,ОФЗ-ПД 26219,fixed,short_fixed_selected
1,SU26226RMFS9,ОФЗ-ПД 26226,fixed,short_fixed_selected
2,SU26207RMFS9,ОФЗ-ПД 26207,fixed,short_fixed_selected
3,SU26232RMFS7,ОФЗ-ПД 26232,fixed,medium_fixed_selected
4,SU26236RMFS8,ОФЗ-ПД 26236,fixed,medium_fixed_selected
5,SU26242RMFS6,ОФЗ-ПД 26242,fixed,medium_fixed_selected
6,SU26240RMFS0,ОФЗ-ПД 26240,fixed,long_fixed_selected
7,SU26243RMFS4,ОФЗ-ПД 26243,fixed,long_fixed_selected
8,SU26238RMFS4,ОФЗ-ПД 26238,fixed,long_fixed_selected
9,SU29024RMFS5,ОФЗ-ПК 29024,floating,floating_coupon


## 3. Загрузка данных из CSV


In [3]:
def read_csv_robust(path: Path) -> pd.DataFrame:
    """Чтение CSV с несколькими возможными кодировками."""
    encodings = ["utf-8-sig", "utf-8", "cp1251"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}: {last_error}")


def load_input_data(input_dir: Path) -> pd.DataFrame:
    combined_path = input_dir / "ofz_all_combined_2016_2026.csv"

    if combined_path.exists():
        print(f"Loading combined file: {combined_path}")
        return read_csv_robust(combined_path)
    csv_files = sorted([
        p for p in input_dir.glob("*.csv")
        if "summary" not in p.name.lower()
        and "combined" not in p.name.lower()
        and "clean" not in p.name.lower()
    ])

    print(f"Combined file not found. Loading {len(csv_files)} individual CSV files...")
    parts = []
    for path in csv_files:
        df_part = read_csv_robust(path)
        df_part["SOURCE_FILE"] = path.name
        parts.append(df_part)

    return pd.concat(parts, ignore_index=True)


df_raw = load_input_data(INPUT_DIR)
print("Raw shape:", df_raw.shape)
display(df_raw.head())
print(df_raw.columns.tolist())


Loading combined file: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/ofz_moex_csv_2016_2026/ofz_all_combined_2016_2026.csv
Raw shape: (15644, 32)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,BONDTYPE,BONDSUBTYPE
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,96.0501,96.0501,95.2501,95.5993,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,NaN,NaN
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.6000,95.8000,95.3550,95.6201,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,NaN,NaN
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8888,95.9000,95.7000,95.9000,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,NaN,NaN
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8000,95.9500,95.2900,95.7000,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,NaN,NaN
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.9499,96.5999,95.7600,96.4500,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,NaN,NaN


['TRADEDATE', 'SECID', 'ISSUE_NAME', 'SHORTNAME', 'BOARDID', 'COUPON_TYPE', 'ANALYSIS_GROUP', 'MATURITY_GROUP', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'WAPRICE', 'CLEAN_PRICE_RUB', 'DIRTY_PRICE_RUB', 'PRICE_DEV_FROM_100', 'PRICE_RETURN', 'YIELDATWAP', 'YIELDCLOSE', 'YIELD', 'VOLUME', 'VALUE', 'NUMTRADES', 'ACCINT', 'FACEVALUE', 'COUPONPERCENT', 'COUPONVALUE', 'MATDATE', 'YEARS_TO_MATURITY', 'DURATION', 'BONDTYPE', 'BONDSUBTYPE']


## 4. Базовая нормализация колонок и типов

На этом шаге:

- приводим даты;
- приводим числовые поля к `float`/`int`;
- добавляем недостающие метаданные из словаря `BONDS`;
- пересчитываем `YIELD`, `YEARS_TO_MATURITY`, `MATURITY_GROUP`, цены в рублях и доходность цены.


In [4]:
DATE_COLUMNS = ["TRADEDATE", "MATDATE"]

NUMERIC_COLUMNS = [
    "OPEN", "HIGH", "LOW", "CLOSE", "WAPRICE",
    "ACCINT", "FACEVALUE", "VOLUME", "VALUE", "NUMTRADES",
    "YIELDATWAP", "YIELDCLOSE", "YIELD",
    "COUPONPERCENT", "COUPONVALUE", "DURATION",
    "CLEAN_PRICE_RUB", "DIRTY_PRICE_RUB", "PRICE_DEV_FROM_100", "PRICE_RETURN",
    "YEARS_TO_MATURITY",
]


def add_missing_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def normalize_base(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    required_base = ["TRADEDATE", "SECID", "CLOSE"]
    for col in required_base:
        if col not in df.columns:
            raise ValueError(f"В данных нет обязательной колонки: {col}")

    for col in DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[(df["TRADEDATE"] >= START_DATE) & (df["TRADEDATE"] <= END_DATE)].copy()
    df["ISSUE_NAME"] = df["SECID"].map(lambda x: BOND_META.get(x, {}).get("ISSUE_NAME", pd.NA))
    df["COUPON_TYPE"] = df["SECID"].map(lambda x: BOND_META.get(x, {}).get("COUPON_TYPE", pd.NA))
    df["ANALYSIS_GROUP"] = df["SECID"].map(lambda x: BOND_META.get(x, {}).get("ANALYSIS_GROUP", pd.NA))
    if "SHORTNAME" not in df.columns:
        df["SHORTNAME"] = df["ISSUE_NAME"]
    if "YIELDATWAP" in df.columns and "YIELDCLOSE" in df.columns:
        df["YIELD"] = df["YIELDATWAP"].fillna(df["YIELDCLOSE"])
    elif "YIELDATWAP" in df.columns:
        df["YIELD"] = df["YIELDATWAP"]
    elif "YIELDCLOSE" in df.columns:
        df["YIELD"] = df["YIELDCLOSE"]
    else:
        df["YIELD"] = pd.NA

    df["YIELD_FOR_ANALYSIS"] = df["YIELD"]
    if "WAPRICE" in df.columns:
        df["PRICE_FOR_ANALYSIS"] = df["WAPRICE"].fillna(df["CLOSE"])
    else:
        df["PRICE_FOR_ANALYSIS"] = df["CLOSE"]
    if "MATDATE" in df.columns:
        df["YEARS_TO_MATURITY"] = (df["MATDATE"] - df["TRADEDATE"]).dt.days / 365.25
    else:
        df["YEARS_TO_MATURITY"] = pd.NA

    def maturity_group(years):
        if pd.isna(years):
            return pd.NA
        if years <= 1:
            return "short"
        if years <= 5:
            return "medium"
        return "long"

    df["MATURITY_GROUP"] = df["YEARS_TO_MATURITY"].map(maturity_group)
    df["PLOT_GROUP"] = np.where(
        df["COUPON_TYPE"].eq("floating"),
        "floating_coupon",
        df["MATURITY_GROUP"].astype("object")
    )

    if {"CLOSE", "FACEVALUE"}.issubset(df.columns):
        df["CLEAN_PRICE_RUB"] = df["CLOSE"] / 100 * df["FACEVALUE"]
    else:
        df["CLEAN_PRICE_RUB"] = pd.NA

    if {"CLEAN_PRICE_RUB", "ACCINT"}.issubset(df.columns):
        df["DIRTY_PRICE_RUB"] = df["CLEAN_PRICE_RUB"] + df["ACCINT"]
    else:
        df["DIRTY_PRICE_RUB"] = pd.NA

    df["PRICE_DEV_FROM_100"] = df["CLOSE"] - 100
    df = df.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)
    df["PRICE_RETURN"] = df.groupby("SECID")["CLOSE"].pct_change()

    return df


df_base = normalize_base(df_raw)
print("Base shape:", df_base.shape)
display(df_base.head())


Base shape: (15644, 35)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,BONDTYPE,BONDSUBTYPE,YIELD_FOR_ANALYSIS,PRICE_FOR_ANALYSIS,PLOT_GROUP
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,96.0501,96.0501,95.2501,95.5993,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,NaN,NaN,8.99,95.5773,long
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.6000,95.8000,95.3550,95.6201,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,NaN,NaN,8.98,95.6335,long
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8888,95.9000,95.7000,95.9000,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,NaN,NaN,8.95,95.7770,long
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8000,95.9500,95.2900,95.7000,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,NaN,NaN,8.96,95.7496,long
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.9499,96.5999,95.7600,96.4500,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,NaN,NaN,8.86,96.3739,long


## 5. Удаление дубликатов и критических ошибок

Критически важные поля для дневной рыночной записи:

- `TRADEDATE` — дата торгов;
- `SECID` — код выпуска;
- `CLOSE` — цена закрытия.

Строки без этих полей не подходят для дальнейшего анализа. Также удаляются очевидные технические ошибки: неположительные цены, отрицательные объёмы/обороты и записи после даты погашения.


In [5]:
def clean_core_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()

    log = []
    n_start = len(df)
    duplicate_mask = df.duplicated(subset=["SECID", "TRADEDATE"], keep="last")
    n_duplicates = int(duplicate_mask.sum())
    df = df[~duplicate_mask].copy()
    critical_missing_mask = df[["TRADEDATE", "SECID", "CLOSE"]].isna().any(axis=1)
    n_critical_missing = int(critical_missing_mask.sum())
    df = df[~critical_missing_mask].copy()
    bad_price_mask = df["CLOSE"] <= 0
    n_bad_price = int(bad_price_mask.sum())
    df = df[~bad_price_mask].copy()

    bad_market_mask = pd.Series(False, index=df.index)
    for col in ["VOLUME", "VALUE", "NUMTRADES"]:
        if col in df.columns:
            bad_market_mask = bad_market_mask | (df[col].notna() & (df[col] < 0))
    n_bad_market = int(bad_market_mask.sum())
    df = df[~bad_market_mask].copy()
    if "YEARS_TO_MATURITY" in df.columns:
        after_maturity_mask = df["YEARS_TO_MATURITY"].notna() & (df["YEARS_TO_MATURITY"] < 0)
    else:
        after_maturity_mask = pd.Series(False, index=df.index)
    n_after_maturity = int(after_maturity_mask.sum())
    df = df[~after_maturity_mask].copy()

    fixed_missing_yield_mask = df["COUPON_TYPE"].eq("fixed") & df["YIELD_FOR_ANALYSIS"].isna()
    n_fixed_missing_yield = int(fixed_missing_yield_mask.sum())
    df = df[~fixed_missing_yield_mask].copy()

    df = df.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)
    df["PRICE_RETURN"] = df.groupby("SECID")["CLOSE"].pct_change()

    log.append({"step": "start_rows", "rows": n_start})
    log.append({"step": "drop_duplicates_SECID_TRADEDATE", "rows": n_duplicates})
    log.append({"step": "drop_critical_missing", "rows": n_critical_missing})
    log.append({"step": "drop_bad_price", "rows": n_bad_price})
    log.append({"step": "drop_bad_market_values", "rows": n_bad_market})
    log.append({"step": "drop_after_maturity", "rows": n_after_maturity})
    log.append({"step": "drop_fixed_missing_yield", "rows": n_fixed_missing_yield})
    log.append({"step": "final_rows_after_core_cleaning", "rows": len(df)})

    return df, pd.DataFrame(log)


df_core, cleaning_log = clean_core_rows(df_base)
print("Core-clean shape:", df_core.shape)
display(cleaning_log)


Core-clean shape: (15522, 35)


,step,rows
0,start_rows,15644
1,drop_duplicates_SECID_TRADEDATE,0
2,drop_critical_missing,122
3,drop_bad_price,0
4,drop_bad_market_values,0
5,drop_after_maturity,0
6,drop_fixed_missing_yield,0
7,final_rows_after_core_cleaning,15522


## 6. Выбросы

Сначала выбросы определяются и помечаются флагами. Затем в следующем блоке найденные выбросы заменяются медианными значениями за тот же торговый день внутри сопоставимой группы облигаций.

Создаются флаги:

- `CLOSE_IQR_OUTLIER` — выброс по цене закрытия;
- `YIELD_IQR_OUTLIER` — выброс по доходности;
- `PRICE_RETURN_IQR_OUTLIER` — выброс по дневному изменению цены;
- `ANY_OUTLIER_FLAG` — есть хотя бы один из этих флагов.


In [6]:
def add_iqr_outlier_flag(
    df: pd.DataFrame,
    group_col: str,
    value_col: str,
    flag_col: str,
    multiplier: float = 3.0,
    min_non_na: int = 30,
) -> pd.DataFrame:
    df = df.copy()
    df[flag_col] = False

    for key, idx in df.groupby(group_col).groups.items():
        values = df.loc[idx, value_col]
        values_non_na = values.dropna()

        if len(values_non_na) < min_non_na:
            continue

        q1 = values_non_na.quantile(0.25)
        q3 = values_non_na.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            continue

        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr

        flag = values.lt(lower) | values.gt(upper)
        df.loc[idx, flag_col] = flag.fillna(False)

    return df


def add_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = add_iqr_outlier_flag(df, "SECID", "CLOSE", "CLOSE_IQR_OUTLIER", multiplier=3.0)
    df = add_iqr_outlier_flag(df, "SECID", "PRICE_RETURN", "PRICE_RETURN_IQR_OUTLIER", multiplier=5.0)
    df = add_iqr_outlier_flag(df, "SECID", "YIELD_FOR_ANALYSIS", "YIELD_IQR_OUTLIER", multiplier=3.0)

    df["HARD_DATA_ERROR"] = False
    df.loc[df["CLOSE"].le(0), "HARD_DATA_ERROR"] = True
    df.loc[df["PRICE_FOR_ANALYSIS"].le(0), "HARD_DATA_ERROR"] = True

    df["YIELD_EXTREME_LEVEL_FLAG"] = df["YIELD_FOR_ANALYSIS"].notna() & (
        df["YIELD_FOR_ANALYSIS"].lt(-10) | df["YIELD_FOR_ANALYSIS"].gt(100)
    )

    outlier_cols = [
        "CLOSE_IQR_OUTLIER",
        "PRICE_RETURN_IQR_OUTLIER",
        "YIELD_IQR_OUTLIER",
        "YIELD_EXTREME_LEVEL_FLAG",
        "HARD_DATA_ERROR",
    ]

    df["ANY_OUTLIER_FLAG"] = df[outlier_cols].any(axis=1)

    return df


df_flags = add_outlier_flags(df_core)

outlier_cols = [
    "CLOSE_IQR_OUTLIER",
    "PRICE_RETURN_IQR_OUTLIER",
    "YIELD_IQR_OUTLIER",
    "YIELD_EXTREME_LEVEL_FLAG",
    "HARD_DATA_ERROR",
    "ANY_OUTLIER_FLAG",
]

outlier_summary = (
    df_flags
    .groupby(["SECID", "ISSUE_NAME"], dropna=False)[outlier_cols]
    .sum()
    .reset_index()
)

display(outlier_summary)


,SECID,ISSUE_NAME,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG
0,SU26207RMFS9,ОФЗ-ПД 26207,0,27,0,0,0,27
1,SU26219RMFS4,ОФЗ-ПД 26219,0,25,0,0,0,25
2,SU26226RMFS9,ОФЗ-ПД 26226,0,27,0,0,0,27
3,SU26232RMFS7,ОФЗ-ПД 26232,0,20,0,0,0,20
4,SU26236RMFS8,ОФЗ-ПД 26236,0,10,0,0,0,10
5,SU26238RMFS4,ОФЗ-ПД 26238,0,13,0,0,0,13
6,SU26240RMFS0,ОФЗ-ПД 26240,0,16,0,0,0,16
7,SU26242RMFS6,ОФЗ-ПД 26242,0,2,0,0,0,2
8,SU26243RMFS4,ОФЗ-ПД 26243,0,5,0,0,0,5
9,SU29024RMFS5,ОФЗ-ПК 29024,0,0,0,0,0,0


## 6.1. Замена выбросов дневной медианой

В этом блоке выбросы заменяются медианными значениями за тот же торговый день внутри сопоставимой группы облигаций:

1. сначала берётся медиана по `TRADEDATE + COUPON_TYPE + MATURITY_GROUP`;
2. если в такой группе медиана недоступна, используется медиана по `TRADEDATE + COUPON_TYPE`;
3. затем медиана по `TRADEDATE`;
4. последний резерв — медиана по самому выпуску `SECID`.


In [7]:
def replace_outliers_with_daily_medians(df: pd.DataFrame) -> pd.DataFrame:
    """Заменяет найденные выбросы медианами за тот же день с групповыми fallback-правилами.

    Логика:
    - ценовые выбросы заменяются для CLOSE, WAPRICE и PRICE_FOR_ANALYSIS;
    - выбросы доходности заменяются для YIELDATWAP, YIELDCLOSE и YIELD_FOR_ANALYSIS;
    - исходные значения сохраняются в *_RAW;
    - после замены пересчитываются производные ценовые признаки.
    """
    df = df.copy()

    raw_cols = [
        "CLOSE", "WAPRICE", "PRICE_FOR_ANALYSIS",
        "YIELDATWAP", "YIELDCLOSE", "YIELD_FOR_ANALYSIS",
    ]
    for col in raw_cols:
        if col in df.columns and f"{col}_RAW" not in df.columns:
            df[f"{col}_RAW"] = df[col]

    price_mask = pd.Series(False, index=df.index)
    for col in ["CLOSE_IQR_OUTLIER", "PRICE_RETURN_IQR_OUTLIER", "HARD_DATA_ERROR"]:
        if col in df.columns:
            price_mask = price_mask | df[col].fillna(False)

    yield_mask = pd.Series(False, index=df.index)
    for col in ["YIELD_IQR_OUTLIER", "YIELD_EXTREME_LEVEL_FLAG"]:
        if col in df.columns:
            yield_mask = yield_mask | df[col].fillna(False)

    def daily_group_median(value_col: str) -> pd.Series:
        med_1 = df.groupby(["TRADEDATE", "COUPON_TYPE", "MATURITY_GROUP"], dropna=False)[value_col].transform("median")

        med_2 = df.groupby(["TRADEDATE", "COUPON_TYPE"], dropna=False)[value_col].transform("median")

        med_3 = df.groupby("TRADEDATE", dropna=False)[value_col].transform("median")

        med_4 = df.groupby("SECID", dropna=False)[value_col].transform("median")

        return med_1.fillna(med_2).fillna(med_3).fillna(med_4)

    price_cols = [col for col in ["CLOSE", "WAPRICE", "PRICE_FOR_ANALYSIS"] if col in df.columns]
    for col in price_cols:
        replacement = daily_group_median(col)
        rows = price_mask & replacement.notna()
        df.loc[rows, col] = replacement.loc[rows]

    yield_cols = [col for col in ["YIELDATWAP", "YIELDCLOSE", "YIELD_FOR_ANALYSIS"] if col in df.columns]
    for col in yield_cols:
        replacement = daily_group_median(col)
        rows = yield_mask & replacement.notna()
        df.loc[rows, col] = replacement.loc[rows]

    df["OUTLIER_REPLACED_WITH_DAILY_MEDIAN"] = price_mask | yield_mask

    if "WAPRICE" in df.columns and "CLOSE" in df.columns:
        df["PRICE_FOR_ANALYSIS"] = df["WAPRICE"].fillna(df["CLOSE"])
    elif "CLOSE" in df.columns:
        df["PRICE_FOR_ANALYSIS"] = df["CLOSE"]

    if {"CLOSE", "FACEVALUE"}.issubset(df.columns):
        df["CLEAN_PRICE_RUB"] = df["CLOSE"] / 100 * df["FACEVALUE"]

    if {"CLEAN_PRICE_RUB", "ACCINT"}.issubset(df.columns):
        df["DIRTY_PRICE_RUB"] = df["CLEAN_PRICE_RUB"] + df["ACCINT"]

    if "CLOSE" in df.columns:
        df["PRICE_DEV_FROM_100"] = df["CLOSE"] - 100
        df = df.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)
        df["PRICE_RETURN"] = df.groupby("SECID")["CLOSE"].pct_change()

    return df


before_replacement = df_flags.copy()
df_flags = replace_outliers_with_daily_medians(df_flags)

replacement_report = (
    df_flags
    .groupby(["SECID", "ISSUE_NAME"], dropna=False)["OUTLIER_REPLACED_WITH_DAILY_MEDIAN"]
    .sum()
    .reset_index(name="replaced_rows")
    .sort_values("replaced_rows", ascending=False)
)

display(replacement_report)
print("Total replaced rows:", int(df_flags["OUTLIER_REPLACED_WITH_DAILY_MEDIAN"].sum()))



,SECID,ISSUE_NAME,replaced_rows
0,SU26207RMFS9,ОФЗ-ПД 26207,27
2,SU26226RMFS9,ОФЗ-ПД 26226,27
1,SU26219RMFS4,ОФЗ-ПД 26219,25
3,SU26232RMFS7,ОФЗ-ПД 26232,20
6,SU26240RMFS0,ОФЗ-ПД 26240,16
5,SU26238RMFS4,ОФЗ-ПД 26238,13
4,SU26236RMFS8,ОФЗ-ПД 26236,10
8,SU26243RMFS4,ОФЗ-ПД 26243,5
11,SU29026RMFS0,ОФЗ-ПК 29026,4
7,SU26242RMFS6,ОФЗ-ПД 26242,2


Total replaced rows: 150


## 7. Финальный набор колонок

Здесь формируется аккуратная дневная таблица. В ней остаются только поля, которые действительно нужны:

- идентификаторы выпуска;
- дата торгов;
- цены;
- доходности;
- объёмы торгов;
- купонные характеристики;
- дата погашения и срок до погашения;
- группы для анализа;
- флаги качества данных.


In [8]:
FINAL_DAILY_COLUMNS = [
    # Идентификаторы
    "TRADEDATE",
    "SECID",
    "ISSUE_NAME",
    "SHORTNAME",
    "BOARDID",

    # Группы анализа
    "COUPON_TYPE",
    "ANALYSIS_GROUP",
    "MATURITY_GROUP",
    "PLOT_GROUP",

    # Цены и ценовая динамика
    "OPEN",
    "HIGH",
    "LOW",
    "CLOSE",
    "WAPRICE",
    "PRICE_FOR_ANALYSIS",
    "CLEAN_PRICE_RUB",
    "DIRTY_PRICE_RUB",
    "PRICE_DEV_FROM_100",
    "PRICE_RETURN",

    # Доходности
    "YIELDATWAP",
    "YIELDCLOSE",
    "YIELD_FOR_ANALYSIS",

    # Объёмы торгов
    "VOLUME",
    "VALUE",
    "NUMTRADES",

    # Купонные и облигационные характеристики
    "ACCINT",
    "FACEVALUE",
    "COUPONPERCENT",
    "COUPONVALUE",
    "MATDATE",
    "YEARS_TO_MATURITY",
    "DURATION",

    # Флаги качества
    "CLOSE_IQR_OUTLIER",
    "PRICE_RETURN_IQR_OUTLIER",
    "YIELD_IQR_OUTLIER",
    "YIELD_EXTREME_LEVEL_FLAG",
    "HARD_DATA_ERROR",
    "ANY_OUTLIER_FLAG",
    "OUTLIER_REPLACED_WITH_DAILY_MEDIAN",

    # Исходные значения до медианной замены выбросов
    "CLOSE_RAW",
    "WAPRICE_RAW",
    "PRICE_FOR_ANALYSIS_RAW",
    "YIELDATWAP_RAW",
    "YIELDCLOSE_RAW",
    "YIELD_FOR_ANALYSIS_RAW",
]

existing_final_columns = [col for col in FINAL_DAILY_COLUMNS if col in df_flags.columns]

df_daily = df_flags[existing_final_columns].copy()

df_daily = df_daily.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

assert not df_daily.duplicated(subset=["SECID", "TRADEDATE"]).any(), "Есть дубликаты SECID + TRADEDATE"
assert df_daily["TRADEDATE"].notna().all(), "Есть пустые TRADEDATE"
assert df_daily["SECID"].notna().all(), "Есть пустые SECID"
assert df_daily["CLOSE"].notna().all(), "Есть пустые CLOSE"
assert (df_daily["CLOSE"] > 0).all(), "Есть неположительные CLOSE"

print("Final daily shape:", df_daily.shape)
display(df_daily.head())
display(df_daily.tail())


Final daily shape: (15522, 45)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,96.0501,96.0501,95.2501,95.5993,95.5773,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,False,False,False,False,False,False,False,95.5993,95.5773,95.5773,8.99,8.98,8.99
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.6000,95.8000,95.3550,95.6201,95.6335,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,False,False,False,False,False,False,False,95.6201,95.6335,95.6335,8.98,8.98,8.98
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8888,95.9000,95.7000,95.9000,95.7770,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,False,False,False,False,False,False,False,95.9000,95.7770,95.7770,8.95,8.94,8.95
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8000,95.9500,95.2900,95.7000,95.7496,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,False,False,False,False,False,False,False,95.7000,95.7496,95.7496,8.96,8.97,8.96
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.9499,96.5999,95.7600,96.4500,96.3739,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,False,False,False,False,False,False,False,96.4500,96.3739,96.3739,8.86,8.85,8.86


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW
15517,2026-04-24,SU29026RMFS0,ОФЗ-ПК 29026,ОФЗ 29026,TQOB,floating,floating_coupon,long,floating_coupon,96.232,96.485,96.120,96.468,96.272,96.272,964.68,985.78,-3.532,0.000000,0.0,NaN,0.0,1977,1903294.15,202,21.10,1000,0.0,0.0,2038-09-04,12.364134,NaN,False,False,False,False,False,False,False,96.468,96.272,96.272,0.0,NaN,0.0
15518,2026-04-27,SU29026RMFS0,ОФЗ-ПК 29026,ОФЗ 29026,TQOB,floating,floating_coupon,long,floating_coupon,96.126,96.595,96.120,96.399,96.351,96.351,963.99,986.30,-3.601,-0.000715,0.0,NaN,0.0,1730,1666868.88,347,22.31,1000,0.0,0.0,2038-09-04,12.355921,NaN,False,False,False,False,False,False,False,96.399,96.351,96.351,0.0,NaN,0.0
15519,2026-04-28,SU29026RMFS0,ОФЗ-ПК 29026,ОФЗ 29026,TQOB,floating,floating_coupon,long,floating_coupon,96.409,96.540,96.138,96.330,96.512,96.512,963.30,986.01,-3.670,-0.000716,0.0,NaN,0.0,9987,9638638.23,167,22.71,1000,0.0,0.0,2038-09-04,12.353183,NaN,False,False,False,False,False,False,False,96.330,96.512,96.512,0.0,NaN,0.0
15520,2026-04-29,SU29026RMFS0,ОФЗ-ПК 29026,ОФЗ 29026,TQOB,floating,floating_coupon,long,floating_coupon,96.468,96.690,96.138,96.299,96.300,96.300,962.99,986.11,-3.701,-0.000322,0.0,NaN,0.0,2576,2480687.86,200,23.12,1000,0.0,0.0,2038-09-04,12.350445,NaN,False,False,False,False,False,False,False,96.299,96.300,96.300,0.0,NaN,0.0
15521,2026-04-30,SU29026RMFS0,ОФЗ-ПК 29026,ОФЗ 29026,TQOB,floating,floating_coupon,long,floating_coupon,96.299,96.404,96.150,96.404,96.196,96.196,964.04,987.56,-3.596,0.001090,0.0,NaN,0.0,1999,1922964.65,204,23.52,1000,0.0,0.0,2038-09-04,12.347707,NaN,False,False,False,False,False,False,False,96.404,96.196,96.196,0.0,NaN,0.0


## 8. Отчёты по качеству данных

In [9]:
def make_bond_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df
        .groupby(["SECID", "ISSUE_NAME", "COUPON_TYPE", "ANALYSIS_GROUP"], dropna=False)
        .agg(
            date_min=("TRADEDATE", "min"),
            date_max=("TRADEDATE", "max"),
            rows=("TRADEDATE", "count"),
            close_na=("CLOSE", lambda x: x.isna().sum()),
            yield_na=("YIELD_FOR_ANALYSIS", lambda x: x.isna().sum()),
            yield_coverage=("YIELD_FOR_ANALYSIS", lambda x: x.notna().mean()),
            close_mean=("CLOSE", "mean"),
            close_min=("CLOSE", "min"),
            close_max=("CLOSE", "max"),
            yield_mean=("YIELD_FOR_ANALYSIS", "mean"),
            yield_min=("YIELD_FOR_ANALYSIS", "min"),
            yield_max=("YIELD_FOR_ANALYSIS", "max"),
            volume_mean=("VOLUME", "mean"),
            value_mean=("VALUE", "mean"),
            numtrades_mean=("NUMTRADES", "mean"),
            outlier_rows=("ANY_OUTLIER_FLAG", "sum"),
        )
        .reset_index()
    )
    return summary


bond_summary = make_bond_summary(df_daily)

missing_values_report = (
    df_daily
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_share"})
)
missing_values_report["missing_count"] = [df_daily[col].isna().sum() for col in missing_values_report["column"]]

outlier_rows = df_daily[df_daily["ANY_OUTLIER_FLAG"]].copy()

print("Bond summary:")
display(bond_summary)

print("Missing values report:")
display(missing_values_report.head(30))

print("Outlier rows:", outlier_rows.shape)
display(outlier_rows.head())


Bond summary:


,SECID,ISSUE_NAME,COUPON_TYPE,ANALYSIS_GROUP,date_min,date_max,rows,close_na,yield_na,yield_coverage,close_mean,close_min,close_max,yield_mean,yield_min,yield_max,volume_mean,value_mean,numtrades_mean,outlier_rows
0,SU26207RMFS9,ОФЗ-ПД 26207,fixed,short_fixed_selected,2016-05-04,2026-04-30,2516,0,0,1.0,99.607235,78.900,117.988,9.715262,5.04,21.67,4.231888e+05,4.217148e+08,738.849364,27
1,SU26219RMFS4,ОФЗ-ПД 26219,fixed,short_fixed_selected,2016-07-06,2026-04-30,2472,0,0,1.0,98.437791,78.900,114.951,9.733794,5.00,21.88,2.757739e+05,2.695215e+08,519.099919,25
2,SU26226RMFS9,ОФЗ-ПД 26226,fixed,short_fixed_selected,2019-02-06,2026-04-30,1817,0,0,1.0,98.564920,78.900,116.166,10.381365,4.99,21.83,3.181275e+05,3.130207e+08,653.223445,27
3,SU26232RMFS7,ОФЗ-ПД 26232,fixed,medium_fixed_selected,2019-12-11,2026-04-30,1602,0,0,1.0,88.631091,63.620,110.014,10.829607,5.18,20.95,2.594203e+05,2.355812e+08,527.500000,20
4,SU26236RMFS8,ОФЗ-ПД 26236,fixed,medium_fixed_selected,2020-11-25,2026-04-30,1364,0,0,1.0,83.100258,63.620,100.625,11.722903,5.66,20.53,3.110622e+05,2.656414e+08,509.798387,10
5,SU26238RMFS4,ОФЗ-ПД 26238,fixed,long_fixed_selected,2021-06-23,2026-04-30,1219,0,0,1.0,68.257124,48.528,100.998,12.118187,7.19,16.65,1.208797e+06,7.270605e+08,9230.154225,13
6,SU26240RMFS0,ОФЗ-ПД 26240,fixed,long_fixed_selected,2021-07-15,2026-04-30,1203,0,0,1.0,70.257911,50.584,99.890,12.366201,7.16,17.36,4.650758e+05,3.217171e+08,2472.173732,16
7,SU26242RMFS6,ОФЗ-ПД 26242,fixed,medium_fixed_selected,2023-01-26,2026-04-30,829,0,0,1.0,85.543609,69.239,100.000,13.880072,9.17,20.01,3.684200e+05,3.128226e+08,1799.517491,2
8,SU26243RMFS4,ОФЗ-ПД 26243,fixed,long_fixed_selected,2023-06-22,2026-04-30,729,0,0,1.0,76.080744,53.498,101.299,14.402428,9.86,17.65,9.069108e+05,6.679859e+08,5275.393690,5
9,SU29024RMFS5,ОФЗ-ПК 29024,floating,floating_coupon,2023-05-04,2026-04-30,762,0,0,1.0,96.120803,91.829,99.069,0.000000,0.00,0.00,8.775627e+04,8.477740e+07,310.937008,0


Missing values report:


,column,missing_share,missing_count
0,YIELDCLOSE_RAW,0.114096,1771
1,YIELDCLOSE,0.114096,1771
2,DURATION,0.114096,1771
3,COUPONPERCENT,0.033114,514
4,COUPONVALUE,0.033114,514
5,PRICE_RETURN,0.000773,12
6,TRADEDATE,0.000000,0
7,PRICE_RETURN_IQR_OUTLIER,0.000000,0
8,FACEVALUE,0.000000,0
9,MATDATE,0.000000,0


Outlier rows: (150, 45)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW
488,2018-04-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,107.090,107.090,104.570,103.3750,103.9270,103.9270,1033.750,1046.030,3.3750,-0.031716,7.47,7.56,7.47,4519626,4.751193e+09,1022,12.28,1000,8.15,40.64,2027-02-03,8.818617,2356.0,False,True,False,False,False,True,True,104.650,105.224,105.224,7.47,7.56,7.47
642,2018-11-14,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.906,98.000,95.756,96.6225,95.9625,95.9625,966.225,986.545,-3.3775,0.005448,8.85,8.72,8.85,2368734,2.299308e+09,652,20.32,1000,8.15,40.64,2027-02-03,8.221766,2188.0,False,True,False,False,False,True,True,97.753,96.996,96.996,8.85,8.72,8.85
972,2020-03-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,108.000,109.899,106.110,106.1345,105.7460,105.7460,1061.345,1067.375,6.1345,-0.045553,6.86,6.82,6.86,6027705,6.485209e+09,947,6.03,1000,8.15,40.64,2027-02-03,6.902122,1989.0,False,True,False,False,False,True,True,107.853,107.655,107.655,6.86,6.82,6.86
974,2020-03-12,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,106.130,106.130,102.000,100.8155,101.8140,101.8140,1008.155,1014.635,0.8155,-0.052575,7.63,7.85,7.63,2177768,2.262513e+09,990,6.48,1000,8.15,40.64,2027-02-03,6.896646,1974.0,False,True,False,False,False,True,True,102.350,103.477,103.477,7.63,7.85,7.63
975,2020-03-13,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,102.304,104.879,102.304,101.9100,101.6160,101.6160,1019.100,1025.800,1.9100,0.010856,7.49,7.46,7.49,1149698,1.199675e+09,627,6.70,1000,8.15,40.64,2027-02-03,6.893908,1976.0,False,True,False,False,False,True,True,104.410,104.232,104.232,7.49,7.46,7.49


## 9. Сохранение чистых дневных данных

In [10]:
path_daily = OUTPUT_DIR / "ofz_clean_daily_2016_2026.csv"
path_datalens_daily = OUTPUT_DIR / "ofz_for_datalens_daily_2016_2026.csv"

path_summary = REPORTS_DIR / "ofz_cleaning_summary_2016_2026.csv"
path_missing = REPORTS_DIR / "ofz_missing_values_report_2016_2026.csv"
path_outliers = REPORTS_DIR / "ofz_outlier_rows_2016_2026.csv"
path_log = REPORTS_DIR / "ofz_cleaning_log_2016_2026.csv"

df_daily.to_csv(path_daily, index=False, encoding="utf-8-sig")

# Этот файл сохраняется как промежуточный дневной датасет.
# Финальные компактные файлы для DataLens формируются в 04_prepare_datalens_datasets.ipynb.
df_daily.to_csv(path_datalens_daily, index=False, encoding="utf-8-sig")

for secid, part in df_daily.groupby("SECID"):
    issue_name = part["ISSUE_NAME"].dropna().iloc[0] if part["ISSUE_NAME"].notna().any() else secid
    safe_issue_name = str(issue_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
    file_path = PER_BOND_CLEAN_DIR / f"{safe_issue_name}_{secid}_clean_daily_2016_2026.csv"
    part.to_csv(file_path, index=False, encoding="utf-8-sig")

bond_summary.to_csv(path_summary, index=False, encoding="utf-8-sig")
missing_values_report.to_csv(path_missing, index=False, encoding="utf-8-sig")
outlier_rows.to_csv(path_outliers, index=False, encoding="utf-8-sig")
cleaning_log.to_csv(path_log, index=False, encoding="utf-8-sig")

print("Saved files:")
for path in [
    path_daily,
    path_datalens_daily,
    path_summary,
    path_missing,
    path_outliers,
    path_log,
]:
    print("-", path.resolve())

print("\nPer-bond clean daily files saved to:", PER_BOND_CLEAN_DIR.resolve())

Saved files:
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed/ofz_clean_daily_2016_2026.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed/ofz_for_datalens_daily_2016_2026.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/ofz_cleaning_summary_2016_2026.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/ofz_missing_values_report_2016_2026.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/ofz_outlier_rows_2016_2026.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/ofz_cleaning_log_2016_2026.csv

Per-bond clean daily files saved to: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed/by_bond_daily
